# Point-SAM forest training on Google Colab

Clones this repo, installs Point-SAM + CUDA ops, downloads the Wytham data, builds crops, then runs the smoke-test training notebook.

**Before anything else:** **Runtime → Change runtime type → GPU** (T4 is fine for a smoke test).

Full write-up: [`docs/colab.md`](../docs/colab.md).

## 0. Options

Set these once, then run the rest top to bottom. Use Drive if you want data and checkpoints to survive a disconnect.

In [12]:
REPO_URL = "https://github.com/koshiyomi/pointsam-training.git"
BRANCH = "jianwei-branch"  # branch that contains scripts/setup_colab.py
USE_DRIVE = True  # True → work under MyDrive/pointsam-training (survives disconnects)
DOWNLOAD_DATA = True  # Zenodo wytham.zip (~3.2 GB)
PREPARE_DATA = True  # build tiles + crops (CPU, tens of minutes)
TRAIN_CONFIG = "configs/experiment/smoke_colab.yaml"  # then isprs_lora_colab.yaml

## 1. Clone (or reopen) the repo

In [14]:
import os
import sys
from pathlib import Path

assert Path("/content").exists(), "This notebook expects Google Colab (/content)."

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive/pointsam-training")
else:
    root = Path("/content/pointsam-training")

if not (root / "src" / "treeseg").is_dir():
    root.parent.mkdir(parents=True, exist_ok=True)
    !git clone -b {BRANCH} --depth 1 "{REPO_URL}" "{root}"
else:
    print("repo already present:", root)

os.chdir(root)
sys.path.insert(0, "src")
assert (root / "scripts" / "setup_colab.py").is_file(), (
    f"{root} is missing scripts/setup_colab.py — clone a branch that has Colab support, "
    "or upload this repo to Drive and set USE_DRIVE=True."
)
print("cwd:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo already present: /content/drive/MyDrive/pointsam-training


AssertionError: /content/drive/MyDrive/pointsam-training is missing scripts/setup_colab.py — clone a branch that has Colab support, or upload this repo to Drive and set USE_DRIVE=True.

## 2. Install Point-SAM, weights, and (optional) data

First run can take a while: `torkit3d` compiles CUDA code, and the Zenodo zip is large. Safe to re-run; finished steps are skipped.

In [ ]:
flags = []
if USE_DRIVE:
    flags.append("--drive")
if DOWNLOAD_DATA:
    flags.append("--download-data")
if PREPARE_DATA:
    flags.append("--prepare-data")

!python scripts/setup_colab.py {" ".join(flags)}

## 3. Smoke-test train

Runs `notebooks/02_train.ipynb` with the config above. After smoke works, change `TRAIN_CONFIG` to `configs/experiment/isprs_lora.yaml` and re-run this cell (or open `02_train` interactively).

In [ ]:
import os
import torch

os.environ["TREESEG_CONFIG"] = TRAIN_CONFIG
print("config:", TRAIN_CONFIG)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Enable a GPU runtime, then Runtime → Restart session and re-run from setup."

%run notebooks/02_train.ipynb

## Next

- Checkpoints and curves land in `results/<run>/` (and under Drive if `USE_DRIVE`).
- Score a run with `notebooks/03_benchmark.ipynb` after setting `TREESEG_CONFIG` to a benchmark YAML.
- TreeX clicks need the separate ICE TreeX install; stem / label clicks work here without it.